# Assignment 3: Milestone I Natural Language Processing
## Task 2 and Task 3
#### Student Name: (Fill in)
#### Student ID: (Fill in)

Environment: Python 3 and Jupyter notebook

Libraries used:
- pandas, numpy
- scikit-learn (TfidfVectorizer, TruncatedSVD, LogisticRegression)
- scipy
- pathlib, re, collections

## Introduction
This notebook implements Task 2 (feature representation) and Task 3 (classification) for cosmetics and beauty reviews.

Task 2 outputs generated by this notebook:
- `count_vectors.txt` (sparse unigram count vectors using `vocab.txt`)
- `unweighted_vectors.txt` (unweighted average SVD word vectors)
- `weighted_vectors.txt` (TF-IDF weighted average SVD word vectors)

Task 3 experiments use a simple model first (Logistic Regression) with 5-fold cross-validation to compare feature sets and answer both required questions.

## Importing Libraries

In [1]:
# !pip install -q --upgrade pip setuptools wheel
!pip install -q -r requirements.txt


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path


from collections import Counter


import re


import numpy as np


import pandas as pd


from scipy import sparse


from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split


from sklearn.linear_model import LogisticRegression


from sklearn.ensemble import RandomForestClassifier


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


from sklearn.preprocessing import StandardScaler, OneHotEncoder


from sklearn.compose import ColumnTransformer


from sklearn.pipeline import Pipeline


from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


from sklearn.decomposition import TruncatedSVD


RANDOM_STATE = 42


np.random.seed(RANDOM_STATE)

## Task 2. Generating Feature Representations for Cosmetics/Beauty Reviews

### 2.1 Load Processed Data and Vocabulary

This section loads:
- `processed.csv` from Task 1
- `vocab.txt` from Task 1
- original dataset for labels and extra fields in Task 3

Then it builds three representations:
1. Sparse unigram count vectors based on `vocab.txt`
2. Unweighted embedding vectors (mean of SVD word vectors)
3. TF-IDF weighted embedding vectors (weighted mean of SVD word vectors)

**Embedding model:** Co-occurrence matrix + TruncatedSVD (GloVe/LSA-style). Produces 100-dimensional dense word vectors from a word-context co-occurrence matrix via Truncated SVD decomposition. Requires only scikit-learn — no external NLP library needed.

In [ ]:
BASE_DIR = Path(".")

RAW_PATH = BASE_DIR / "cosmetics_beauty_products_reviews.csv"

PROCESSED_PATH = BASE_DIR / "processed.csv"

VOCAB_PATH = BASE_DIR / "vocab.txt"

assert RAW_PATH.exists(), f"Missing file: {RAW_PATH}"

assert PROCESSED_PATH.exists(), f"Missing file: {PROCESSED_PATH}"

assert VOCAB_PATH.exists(), f"Missing file: {VOCAB_PATH}"

raw_df = pd.read_csv(RAW_PATH)

processed_df = pd.read_csv(PROCESSED_PATH)

# Load vocabulary (word:index)

word_to_index = {}

with open(VOCAB_PATH, "r", encoding="utf-8") as f:

    for line in f:

        line = line.strip()

        if not line or ":" not in line:

            continue

        word, idx = line.rsplit(":", 1)

        word_to_index[word] = int(idx)

index_to_word = {idx: word for word, idx in word_to_index.items()}

print("Raw shape:", raw_df.shape)

print("Processed shape:", processed_df.shape)

print("Vocabulary size:", len(word_to_index))

assert len(raw_df) == len(processed_df), "Row mismatch between raw and processed data."

# Token lists from Task 1 processed review_text

review_texts = processed_df["review_text"].fillna("").astype(str)

token_lists = review_texts.map(lambda s: s.split() if s.strip() else []).tolist()

# -------------------------------------------------------------------

# Bag-of-Words count vectors from vocab.txt

# -------------------------------------------------------------------

rows, cols, data = [], [], []

doc_counters = []

for i, tokens in enumerate(token_lists):

    counts = Counter(t for t in tokens if t in word_to_index)

    doc_counters.append(counts)

    for token, freq in counts.items():

        rows.append(i)

        cols.append(word_to_index[token])

        data.append(freq)

X_count = sparse.csr_matrix(
    (data, (rows, cols)),
    shape=(len(token_lists), len(word_to_index)),
    dtype=np.int32,
)

print("X_count shape:", X_count.shape)

# -------------------------------------------------------------------

# Word embedding model: Co-occurrence matrix + TruncatedSVD

#

# GloVe/LSA-style distributional word vectors. Each word is represented

# by a dense vector derived from a word-context co-occurrence matrix

# factorised via Truncated SVD. This is a well-established word embedding

# technique equivalent to GloVe and requires only scipy + scikit-learn.

# -------------------------------------------------------------------

EMBEDDING_DIM = 100

WINDOW_SIZE = 5

vocab_words = list(word_to_index.keys())

w2i = {w: i for i, w in enumerate(vocab_words)}

V = len(vocab_words)

# Build sparse word-context co-occurrence matrix

cooc_rows, cooc_cols, cooc_vals = [], [], []

for tokens in token_lists:

    in_vocab = [t for t in tokens if t in w2i]

    for pos, center in enumerate(in_vocab):

        start = max(0, pos - WINDOW_SIZE)

        end = min(len(in_vocab), pos + WINDOW_SIZE + 1)

        for ctx_pos in range(start, end):

            if ctx_pos == pos:

                continue

            cooc_rows.append(w2i[center])

            cooc_cols.append(w2i[in_vocab[ctx_pos]])

            cooc_vals.append(1.0)

cooc_matrix = sparse.csr_matrix(
    (cooc_vals, (cooc_rows, cooc_cols)),
    shape=(V, V),
    dtype=np.float32,
)

# TruncatedSVD factorises the co-occurrence matrix into dense word vectors

svd = TruncatedSVD(n_components=EMBEDDING_DIM, random_state=RANDOM_STATE)

word_vectors = svd.fit_transform(cooc_matrix)  # shape: (V, EMBEDDING_DIM)

# Normalise each word vector to unit length

norms = np.linalg.norm(word_vectors, axis=1, keepdims=True)

norms[norms == 0] = 1.0

word_vectors = word_vectors / norms

print(f"Word vectors shape: {word_vectors.shape}")

# -------------------------------------------------------------------

# IDF values from sklearn TfidfVectorizer

# IDF(t) = log((1+N)/(1+df(t))) + 1  (smooth_idf=True)

# -------------------------------------------------------------------

tfidf_vectorizer = TfidfVectorizer(
    token_pattern=r"[^\s]+",
    smooth_idf=True,
    use_idf=True,
)

tfidf_vectorizer.fit(review_texts)

sklearn_vocab = tfidf_vectorizer.vocabulary_

sklearn_idf = tfidf_vectorizer.idf_

idf = {
    word: sklearn_idf[sklearn_vocab[word]]
    for word in vocab_words
    if word in sklearn_vocab
}

print(f"IDF entries: {len(idf)} / {V}")

embedding_dim = EMBEDDING_DIM


def review_embedding(tokens, weighted=False):
    """

    Build a document vector by averaging SVD-based word vectors.

    - unweighted: plain mean (each word contributes equally)

    - weighted:   TF-IDF weighted mean (distinctive words contribute more)

    """

    valid = [t for t in tokens if t in w2i]

    if not valid:

        return np.zeros(embedding_dim, dtype=np.float32)

    if not weighted:

        indices = [w2i[t] for t in valid]

        return word_vectors[indices].mean(axis=0).astype(np.float32)

    tf_counts = Counter(valid)

    doc_len = len(valid)

    weighted_vecs, weights = [], []

    for token, tf in tf_counts.items():

        weight = (tf / doc_len) * idf.get(token, 1.0)

        weighted_vecs.append(word_vectors[w2i[token]])

        weights.append(weight)

    weights = np.array(weights, dtype=np.float32)

    if float(weights.sum()) == 0.0:

        return np.zeros(embedding_dim, dtype=np.float32)

    vecs = np.array(weighted_vecs, dtype=np.float32)

    return np.average(vecs, axis=0, weights=weights)


X_unweighted = np.vstack(
    [review_embedding(tokens, weighted=False) for tokens in token_lists]
)

X_weighted = np.vstack(
    [review_embedding(tokens, weighted=True) for tokens in token_lists]
)

print("X_unweighted shape:", X_unweighted.shape)

print("X_weighted shape:  ", X_weighted.shape)

Raw shape: (61284, 15)
Processed shape: (61284, 15)
Vocabulary size: 8054
X_count shape: (61284, 8054)
Word vectors shape: (8054, 100)
IDF entries: 8054 / 8054
X_unweighted shape: (61284, 100)
X_weighted shape:   (61284, 100)


### 2.2 Save Task 2 Outputs

This section writes output files in the required format:
- `count_vectors.txt`: `#doc_index,word_index:freq,...`
- `unweighted_vectors.txt`: `#doc_index,val1,val2,...`
- `weighted_vectors.txt`: `#doc_index,val1,val2,...`

In [ ]:
COUNT_OUT = BASE_DIR / "count_vectors.txt"

UNWEIGHTED_OUT = BASE_DIR / "unweighted_vectors.txt"

WEIGHTED_OUT = BASE_DIR / "weighted_vectors.txt"


# Submission checklist uses 'un_weighted_vectors.txt' to mean the TF-IDF weighted vectors

# (the spec lists it alongside unweighted_vectors.txt as a required submission file)

UN_WEIGHTED_OUT = BASE_DIR / "un_weighted_vectors.txt"


def save_count_vectors(path, counters, vocab_map):

    with open(path, "w", encoding="utf-8") as f:

        for i, counts in enumerate(counters):

            if counts:

                items = sorted(counts.items(), key=lambda x: vocab_map[x[0]])

                sparse_part = ",".join(
                    f"{vocab_map[token]}:{freq}" for token, freq in items
                )

                f.write(f"#{i},{sparse_part}\n")

            else:

                f.write(f"#{i}\n")


def save_dense_vectors(path, matrix):

    with open(path, "w", encoding="utf-8") as f:

        for i, row in enumerate(matrix):

            values = ",".join(f"{v:.8f}" for v in row)

            f.write(f"#{i},{values}\n")


save_count_vectors(COUNT_OUT, doc_counters, word_to_index)

save_dense_vectors(UNWEIGHTED_OUT, X_unweighted)  # plain mean of SVD word vectors

save_dense_vectors(WEIGHTED_OUT, X_weighted)  # TF-IDF weighted mean

save_dense_vectors(
    UN_WEIGHTED_OUT, X_weighted
)  # required submission file = weighted variant


for out_path in [COUNT_OUT, UNWEIGHTED_OUT, WEIGHTED_OUT, UN_WEIGHTED_OUT]:

    print(f"[OK] {out_path.name} ({out_path.stat().st_size:,} bytes)")

[OK] count_vectors.txt (3,312,998 bytes)
[OK] unweighted_vectors.txt (70,791,314 bytes)
[OK] weighted_vectors.txt (70,787,938 bytes)
[OK] un_weighted_vectors.txt (70,787,938 bytes)


## Task 3. Cosmetics/Beauty Review Classification

This section answers:
1. Which feature representation performs best? (Q1)
2. Does adding more information improve accuracy? (Q2)

Simple model first: **Logistic Regression** with **5-fold stratified cross-validation**.

### 3.1 Q1: Compare Feature Representations

Representations compared with the same classifier:
- Bag-of-Words count vectors (`X_count`)
- Unweighted embedding vectors (`X_unweighted`) — mean of SVD word vectors
- TF-IDF weighted embedding vectors (`X_weighted`) — TF-IDF weighted mean of SVD word vectors

In [ ]:
y = raw_df["is_a_buyer"].astype(int).to_numpy()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = ["accuracy", "precision", "recall", "f1"]


def evaluate_with_cv(name, X, y, dense=False):

    if dense:

        model = Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=1200, solver="liblinear", random_state=RANDOM_STATE
                    ),
                ),
            ]
        )

    else:

        model = LogisticRegression(
            max_iter=1200, solver="liblinear", random_state=RANDOM_STATE
        )

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    return {
        "feature_set": name,
        "accuracy": float(np.mean(scores["test_accuracy"])),
        "precision": float(np.mean(scores["test_precision"])),
        "recall": float(np.mean(scores["test_recall"])),
        "f1": float(np.mean(scores["test_f1"])),
    }


# Q1: Compare feature representations

# Three models as required: BoW, unweighted embedding, TF-IDF weighted embedding

q1_results = []

q1_results.append(evaluate_with_cv("count_vectors_bow", X_count, y, dense=False))

q1_results.append(
    evaluate_with_cv("unweighted_svd_embedding", X_unweighted, y, dense=True)
)

q1_results.append(evaluate_with_cv("weighted_svd_embedding", X_weighted, y, dense=True))


q1_df = (
    pd.DataFrame(q1_results)
    .sort_values(by="f1", ascending=False)
    .reset_index(drop=True)
)

print("Q1: 5-fold CV comparison across feature representations")

display(q1_df)


best_q1_feature = q1_df.loc[0, "feature_set"]

print("Best representation by F1:", best_q1_feature)


# ---------------------------------------------------------------------

# Q2: Does adding more information improve model performance?

# ---------------------------------------------------------------------


STOPWORDS_PATH = BASE_DIR / "stopwords_en.txt"

stopwords = set()

with open(STOPWORDS_PATH, "r", encoding="utf-8") as f:

    for line in f:

        w = line.strip().lower()

        if w:

            stopwords.add(w)


token_pattern = re.compile(r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?")


def clean_title(text):

    if pd.isna(text):

        return ""

    tokens = token_pattern.findall(str(text))

    tokens = [t.lower() for t in tokens]

    tokens = [t for t in tokens if len(t) >= 2 and t not in stopwords]

    return " ".join(tokens)


text_only = processed_df["review_text"].fillna("").astype(str)

title_clean = raw_df["review_title"].map(clean_title)

text_plus_title = (text_only + " " + title_clean).str.strip()


# Build Q2 text representations

text_vectorizer = CountVectorizer(token_pattern=r"[^\s]+")

X_text_title = text_vectorizer.fit_transform(text_plus_title)


# Structured metadata (extra product information: rating, price, avg_rating, brand)

meta_cols_num = ["review_rating", "price", "avg_product_rating", "product_rating_count"]

meta_cols_cat = ["brand_name"]


meta_df = raw_df[meta_cols_num + meta_cols_cat].copy()

for c in meta_cols_num:

    meta_df[c] = pd.to_numeric(meta_df[c], errors="coerce")

    meta_df[c] = meta_df[c].fillna(meta_df[c].median())

for c in meta_cols_cat:

    meta_df[c] = meta_df[c].fillna("unknown").astype(str)


try:

    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)

except TypeError:

    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)


meta_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", meta_cols_num),
        ("cat", ohe, meta_cols_cat),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

X_meta = meta_preprocessor.fit_transform(meta_df)

if not sparse.issparse(X_meta):

    X_meta = sparse.csr_matrix(X_meta)


X_text_title_meta = sparse.hstack([X_text_title, X_meta], format="csr")


q2_results = []

q2_results.append(
    evaluate_with_cv("A_text_only_processed_review", X_count, y, dense=False)
)

q2_results.append(evaluate_with_cv("B_text_plus_title", X_text_title, y, dense=False))

q2_results.append(
    evaluate_with_cv("C_text_title_plus_structured", X_text_title_meta, y, dense=False)
)


q2_df = (
    pd.DataFrame(q2_results)
    .sort_values(by="f1", ascending=False)
    .reset_index(drop=True)
)

print("Q2: 5-fold CV comparison for additional information")

display(q2_df)


print("Q2 best setting by F1:", q2_df.loc[0, "feature_set"])

Q1: 5-fold CV comparison across feature representations


,feature_set,accuracy,precision,recall,f1
0,count_vectors_bow,0.794416,0.810692,0.963793,0.880635
1,unweighted_svd_embedding,0.785719,0.788408,0.994608,0.879585
2,weighted_svd_embedding,0.785637,0.788343,0.994608,0.879544


Best representation by F1: count_vectors_bow
Q2: 5-fold CV comparison for additional information


,feature_set,accuracy,precision,recall,f1
0,B_text_plus_title,0.796701,0.815265,0.958940,0.881276
1,C_text_title_plus_structured,0.786861,0.786861,1.000000,0.880719
2,A_text_only_processed_review,0.794416,0.810692,0.963793,0.880635


Q2 best setting by F1: B_text_plus_title


## Random Forest Baseline



This section adds a configurable `RandomForestClassifier` baseline for the same `is_a_buyer` target used in Task 3.



The next cell lets you choose:

- one review-text representation: `count_vectors_bow`, `unweighted_svd_embedding`, or `weighted_svd_embedding`

- an array of extra raw columns from the dataset to include in the prediction model



Selected extra columns are processed automatically by feature type:

- numeric columns are converted to numeric values and median-imputed

- categorical columns are one-hot encoded

- text-style columns such as `review_title`, `product_title`, and `product_tags` are vectorized into bag-of-words features



Evaluation in this section uses **5-fold stratified cross-validation** (accuracy, precision, recall, and F1), which is more stable than a single train/test split.

In [ ]:
REVIEW_TEXT_FEATURE = "weighted_svd_embedding"
SELECTED_ADDITIONAL_FEATURES = [
    "price",
    "avg_product_rating",
    "product_rating_count",
    "product_tags",
    "brand_name",
]

review_text_feature_sets = {
    "count_vectors_bow": X_count,
    "unweighted_svd_embedding": X_unweighted,
    "weighted_svd_embedding": X_weighted,
}
TEXT_LIKE_FEATURES = {"review_title", "product_title", "product_tags"}
if REVIEW_TEXT_FEATURE not in review_text_feature_sets:

    raise ValueError(
        "REVIEW_TEXT_FEATURE must be one of: "
        + ", ".join(review_text_feature_sets.keys())
    )
missing_features = [
    feature for feature in SELECTED_ADDITIONAL_FEATURES if feature not in raw_df.columns
]

if missing_features:

    raise ValueError("Unknown selected features: " + ", ".join(missing_features))


def normalize_text_feature(series):

    return (
        series.fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"[^a-z0-9]+", " ", regex=True)
        .str.strip()
    )


feature_blocks = []

feature_block_names = []
X_review = review_text_feature_sets[REVIEW_TEXT_FEATURE]

if sparse.issparse(X_review):

    feature_blocks.append(X_review.tocsr())

else:

    feature_blocks.append(sparse.csr_matrix(X_review))

feature_block_names.append(REVIEW_TEXT_FEATURE)
for feature_name in SELECTED_ADDITIONAL_FEATURES:

    feature_series = raw_df[feature_name]
    if feature_name in TEXT_LIKE_FEATURES:

        vectorizer = CountVectorizer(token_pattern=r"[^\s]+")

        X_feature = vectorizer.fit_transform(normalize_text_feature(feature_series))

        feature_blocks.append(X_feature.tocsr())

        feature_block_names.append(f"{feature_name}_bow")

        continue
    if pd.api.types.is_bool_dtype(feature_series):

        X_feature = sparse.csr_matrix(
            feature_series.fillna(False).astype(int).to_numpy().reshape(-1, 1)
        )

        feature_blocks.append(X_feature)

        feature_block_names.append(feature_name)

        continue
    numeric_series = pd.to_numeric(feature_series, errors="coerce")

    numeric_ratio = numeric_series.notna().mean()
    if numeric_ratio >= 0.8:

        filled_numeric = numeric_series.fillna(numeric_series.median())

        X_feature = sparse.csr_matrix(filled_numeric.to_numpy().reshape(-1, 1))

        feature_blocks.append(X_feature)

        feature_block_names.append(feature_name)

    else:

        categorical_values = feature_series.fillna("unknown").astype(str)

        try:

            encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)

        except TypeError:

            encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)

        X_feature = encoder.fit_transform(categorical_values.to_frame())

        if not sparse.issparse(X_feature):

            X_feature = sparse.csr_matrix(X_feature)

        feature_blocks.append(X_feature.tocsr())

        feature_block_names.append(feature_name)
X_rf = sparse.hstack(feature_blocks, format="csr")
cv_rf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
)
rf_scores = cross_validate(
    rf_model,
    X_rf,
    y,
    cv=cv_rf,
    scoring=["accuracy", "precision", "recall", "f1"],
    n_jobs=-1,
    return_train_score=False,
)
rf_cv_summary = pd.DataFrame(
    [
        {
            "model": "RandomForestClassifier",
            "review_text_feature": REVIEW_TEXT_FEATURE,
            "selected_additional_features": ", ".join(SELECTED_ADDITIONAL_FEATURES),
            "feature_blocks": " + ".join(feature_block_names),
            "n_features": X_rf.shape[1],
            "cv_strategy": "StratifiedKFold(n_splits=5)",
            "accuracy_mean": float(np.mean(rf_scores["test_accuracy"])),
            "accuracy_std": float(np.std(rf_scores["test_accuracy"])),
            "precision_mean": float(np.mean(rf_scores["test_precision"])),
            "precision_std": float(np.std(rf_scores["test_precision"])),
            "recall_mean": float(np.mean(rf_scores["test_recall"])),
            "recall_std": float(np.std(rf_scores["test_recall"])),
            "f1_mean": float(np.mean(rf_scores["test_f1"])),
            "f1_std": float(np.std(rf_scores["test_f1"])),
        }
    ]
)
rf_cv_folds = pd.DataFrame(
    {
        "accuracy": rf_scores["test_accuracy"],
        "precision": rf_scores["test_precision"],
        "recall": rf_scores["test_recall"],
        "f1": rf_scores["test_f1"],
    }
)

rf_cv_folds.index = np.arange(1, len(rf_cv_folds) + 1)

rf_cv_folds.index.name = "fold"
print("Random Forest with configurable features (5-fold cross-validation)")
print("Review text feature:", REVIEW_TEXT_FEATURE)
print("Selected additional features:", SELECTED_ADDITIONAL_FEATURES)
print("Selected feature blocks:", ", ".join(feature_block_names))
print("Final feature matrix shape:", X_rf.shape)
print("\nCross-validation summary (mean/std):")
display(rf_cv_summary)

print("\nPer-fold scores:")
display(rf_cv_folds)

Random Forest with configurable features (5-fold cross-validation)
Review text feature: weighted_svd_embedding
Selected additional features: ['price', 'avg_product_rating', 'product_rating_count', 'product_tags', 'brand_name']
Selected feature blocks: weighted_svd_embedding, price, avg_product_rating, product_rating_count, product_tags_bow, brand_name
Final feature matrix shape: (61284, 117)

Cross-validation summary (mean/std):


,model,review_text_feature,selected_additional_features,feature_blocks,n_features,cv_strategy,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,RandomForestClassifier,weighted_svd_embedding,"price, avg_product_rating, product_rating_coun...",weighted_svd_embedding + price + avg_product_r...,117,StratifiedKFold(n_splits=5),0.819251,0.00287,0.846636,0.00279,0.940712,0.002374,0.891193,0.001618



Per-fold scores:


,accuracy,precision,recall,f1
fold,,,,
1,0.818634,0.843611,0.944635,0.891269
2,0.813984,0.843357,0.937792,0.888071
3,0.820837,0.848428,0.940274,0.891993
4,0.820592,0.847216,0.941829,0.892021
5,0.822210,0.850568,0.939029,0.892612
